# C. elegans — Prospective Graph-State Comparison: fastC-GC

Prerequisite: the `cgc` connectivity stage must already exist for the selected `RUN_ID`.
Parallel lane: yes, independent across method namespaces.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.

Compare one-state and continuous baselines with K-means and regularized mixtures using inner selection and frozen held-out evaluation. Raw-history purging separates training and future graph targets. Label-permutation tests do not establish neural regimes.


In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


Notebook interpreter: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/bin/python
Installed backends: {'fastcgc': '0.1.0', 'tigramite': '5.2.10.1', 'scikit-learn': '1.9.0', 'kmedoids': '0.5.5', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}


In [2]:
from effectome.dynamics import GraphStateConfig, TransitionConfig
from effectome.dynamics.state_evaluation import StateEvaluationConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_graph_states_and_transitions,
)

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "cgc"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
FORCE = False

GRAPH_CFG = GraphStateConfig(
    n_states=1, feature_representation="signed_lag_offdiag", assignment="kmeans",
    metric="frobenius", pca_components=8, analysis_scope="prospective", seed=42,
)
TRANSITION_CFG = TransitionConfig(laplace=1.0, n_null=200, heldout_fraction=0.3, seed=42)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} graph states and transitions",
    expected_stages=["state_evaluation", "graph_states", "transitions"],
    force=FORCE,
)


/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cgc graph states and transitions
run_id=reference method=cgc
run_dir=/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/cgc
resume mode: FORCE=False, matching completed checkpoints will be reused
Rerun interrupted notebooks from the top; completed stages resume from checkpoints.
Stage status for reference/cgc:
  - community: completed (signature=b19266cb0c, artifact=yes)
  - connectivity: completed (signature=867a8d53f5, artifact=yes)
  - connectivity.chunk-00000: completed (signature=1b91e06b3b, artifact=yes)
  - connectivity.chunk-00001: completed (signature=1c728c3716, artifact=yes)
  - connectivity.chunk-00002: completed (signature=3b9a51a36a, artifact=yes)
  - connectivity.chunk-00003: completed (signature=fc4c5d2507, artifact=yes)
  - connectivity.chunk-00004: completed (signature=5d38173532, artifact=yes)
  - connectivity.chunk-00005: completed (signature=d6e22c9982, artifact=yes)
 

In [3]:
states, transitions = run_graph_states_and_transitions(
    run,
    graph_cfg=GRAPH_CFG,
    transition_cfg=TRANSITION_CFG,
    selection_cfg=StateEvaluationConfig(graph=GRAPH_CFG),
    analysis_scope="prospective",
    on_unavailable="return",
    force=FORCE,
)
if states is None:
    print("State evaluation unavailable:", run.load_completed("state_evaluation").summary())
else:
    print(states.labels.shape, states.metric, states.silhouette)
    print(transitions.transition_matrix)
    print("Conditional label-sequence diagnostic only:", transitions.p_value)
print("Neural regimes remain unvalidated without independent raw-process calibration.")
print_stage_status(run)


[reference/cgc] state_evaluation: checking checkpoint; computing if needed...
[reference/cgc] state_evaluation: completed in 8.5s
[reference/cgc] graph_states: checking checkpoint; computing if needed...
[reference/cgc] graph_states: completed in 0.0s
[reference/cgc] transitions: checking checkpoint; computing if needed...
[reference/cgc] transitions: completed in 0.0s
Selected graph summary: kmeans 4
Held-out metrics: {'heldout_mse': 2189037.5103124734, 'constant_mse': 2188795.715370989, 'continuous_mse': 4350259.764752679, 'relative_gain': -0.0001104693963837916, 'mean_membership_entropy': 0.0, 'forecast_pair_count': 41.0}
Regime claims require raw-process calibration; label tests concern label sequences.
(186,) frobenius 0.2809961049797123
[[0.86666667 0.06666667 0.03333333 0.03333333]
 [0.05       0.85       0.05       0.05      ]
 [0.03333333 0.03333333 0.86666667 0.06666667]
 [0.06060606 0.03030303 0.03030303 0.87878788]]
Conditional label-sequence diagnostic only: nan
Neural reg